# Zagadnienie 1: Szukanie miejsc zerowych. Implementacja metody Laguerre'a.
### Konrad Słotta 277442

Rozpatrzmy wielomian
$$
w(z) = a_0z^n + a_1z^{n-1} + \cdots + a_{n-1}z + a_n, \quad a_i, z \in \mathbb{C}.
$$
Zauważmy, że wielomian $w(z)$ ma dokładnie $n$ pierwiastków. Do ich znalezienia wykorzystamy metodę Laguerre'a.


### Funkcja obliczająca pochodną wielomianu
Argumenty: wsp - współczynniki wyjściowego wielomianu.

Zwraca wektor współczynników pochodnej wielomianu.

In [65]:
function pochodna(wsp::Vector)
    n = length(wsp) - 1
    
    if n < 0
        return Float64[]
    elseif n == 0
        return [0.]
    end

    wynik = similar(wsp, n)
    
    for i in 1:n
        wynik[i] = wsp[i] * (n - i + 1)
    end
    
    return wynik
end

pochodna (generic function with 2 methods)

### Funkcja wyliczająca wartość wielomianu w punkcie metodą Hornera

Argumenty: wsp - współczynniki wielomianu, z - argument, dla którego obliczana jest wartość wielomianu.

Zwraca wartość wielomianu w zadanym punkcie.

In [66]:
function horner(wsp::Vector, z::Complex)
    wynik = wsp[1]
    
    for i in 2:length(wsp)
        wynik = wynik * z + wsp[i]
    end
    
    return wynik
end

horner (generic function with 2 methods)

### Funkcja znajdująca jeden z pierwiastków wielomianu metodą Laguerre'a
Argumenty: 
- wsp - współczynniki wielomianu,
- z0 - punkt startowy, od którego zaczynamy szukanie pierwiastków,
- eps - dokładność, z jaką szukamy miejsca zerowego,
- max_iter - maksymalna ilość iteracji funkcji.

Zwraca znaleziony pierwiastek wielomianu.

In [68]:
function laguerre_1pierwiastek(wsp::Vector{}, z0=0+0im, eps=1e-12, max_iter=100)
    n = length(wsp) - 1
    z = z0
    d1_wsp = pochodna(wsp)
    d2_wsp = pochodna(d1_wsp)
    for _ in 1:max_iter

        w = horner(wsp, z)
        d1 = horner(d1_wsp, z)
        d2 = horner(d2_wsp, z)

        if abs(w) < eps
            return z
        end

        H = (n - 1) * ((n - 1) * d1^2 - n * w * d2)

        denom1 = d1 +  √H
        denom2 = d1 - √H
        if abs(denom1) > abs(denom2)
            a = n * w / denom1
        else
            a = n * w / denom2
        end

        z_new = z - a

        if abs(z_new - z) < eps
            return z_new
        end
        z = z_new
    end
    
    return z
end

laguerre_1pierwiastek (generic function with 12 methods)

### Funkcja usuwająca zera wiodące z listy

Argumenty: p - wektor liczb

Zwraca listę z usuniętymi zerami wiodącymi (bez zer na początku listy).

In [69]:
function trim_leading_zeros(p::Vector)
    idx = findfirst(x -> x != 0, p)
    return idx === nothing ? [0] : p[idx:end]
end

trim_leading_zeros (generic function with 1 method)

### Funkcja dzieląca wielomian przez dwumian schematem Hornera

Argumenty:
- p - wektor współczynników wielomianu
- root - liczba $a$ definiująca dwumian $(x - a)$, przez który wielomian zostanie podzielony

Zwraca:
- q - wektor współczynników podzielonego wielomianu,
- r - reszta wielomianu.

In [70]:
function horner_div(p::Vector, root::Complex)
    n = length(p)
    q = zeros(ComplexF64, n-1)
    
    q[1] = p[1]
    for i in 2:(n-1)
        q[i] = p[i] + root*q[i-1]
    end
    r = p[end] + root*q[end]
    return q, r
end

horner_div (generic function with 1 method)

### Funkcja znajdująca wszystkie pierwiastki wielomianu wykorzystując metodę Laguerre'a
Algorytm tej funkci opiera się na znalezieniu jednego pierwiastka metodą Laguerre'a, a następnie dzieleniu wielomianu schematem Hornera. Dla podzielonego wielomianu ponownie szukamy pierwiastka metodą Laguerre'a. Dzielimy wielomian w ten sposób, dopóki nie dostaniemy dwumianu. W ten sposób zwracamy listę wszystkich pierwiastków.

Argumenty:
- wsp - współczynniki wielomianu,
- eps - dokładność, z jaką szukamy miejsc zerowych.

Zwraca wektor pierwiastków wielomianu.

In [71]:
function laguerre(wsp::Vector, eps=1e-12)
    wsp = Complex.(wsp)
    pierwiastki = Complex[]
    obecne_wsp = copy(wsp)

    while length(obecne_wsp) > 2
        pierwiastek = laguerre_1pierwiastek(obecne_wsp, 0.0 + 0.0im, eps)
        push!(pierwiastki, pierwiastek)

        q, r = horner_div(obecne_wsp, pierwiastek)
        obecne_wsp = trim_leading_zeros(q)
    end

    a, b = obecne_wsp
    push!(pierwiastki, -b / a)

    return pierwiastki
end

laguerre (generic function with 2 methods)

In [80]:
Wsp = [1im, 1im]
println(laguerre(Wsp))

Complex[-1.0 - 0.0im]
